In [ ]:
# lets you automatically reload modules when their source code changes
%load_ext autoreload
# reload all modules (except those excluded by %aimport) every time before executing the Python code
%autoreload 2
# allows you to display matplotlib plots directly in the notebook
%matplotlib inline

In [ ]:
import os, sys
import tempfile
import random

import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
import sparrow as sp
import spatialdata as sd
import pooch
from sparrow.datasets.registry import get_registry

In [ ]:
font_size = 8
plot_dpi = 250
save_dpi = 500
# Setting fonts, line widths, and figure resolution for all plots in this tutorial:
mpl.rcParams.update({
    "font.family": "serif",
    "axes.titlesize": font_size + 3,
    "axes.titleweight": "bold",
    "axes.labelsize": font_size,
    "xtick.labelsize": font_size,
    "ytick.labelsize": font_size,
    "lines.linewidth": 1.5,
    "figure.dpi": plot_dpi,
})

In [ ]:
ZARR_PATH = "40k_subset_0.zarr"

sdata = sd.read_zarr(ZARR_PATH)

print(sdata)

In [ ]:
# Specify the coordinates for the region of interest (ROI) in the format [x_min, x_max, y_min, y_max]
coordinates = [21000, 25000, 22000, 26000]
channels = ["DAPI", "PolyT"]
z_slices = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5]

In [ ]:
for z in z_slices:
    for ch in channels:
        print(f"z = {z}, channel = {ch}")

        sp.pl.plot_shapes(
            sdata,
            img_layer=["Yujin_vizgen_Liver1Slice1_z_global"],
            crd=coordinates,
            channel=ch,
            z_slice=z,
            figsize=(10, 10),
            fig_kwargs={"dpi": plot_dpi},
        )

In [ ]:
# z layer
z_to_show = 3.5

# input layer
INPUT_LAYER = "Yujin_vizgen_Liver1Slice1_z_global"

In [ ]:
# DAPI values to test
dapi_values = [30, 80, 135, 250]

In [ ]:
for dapi_size in dapi_values:
    layer_name = f"minmax_dapi_{dapi_size}"

    print(f"DAPI filter size = {dapi_size}")

    sdata = sp.im.min_max_filtering(
        sdata=sdata,
        img_layer=INPUT_LAYER,
        output_layer=layer_name,
        crd=coordinates,
        size_min_max_filter=[dapi_size, 285],
        chunks=4096,
        overwrite=True,
    )

    sp.pl.plot_shapes(
        sdata,
        img_layer=[layer_name],
        crd=coordinates,
        channel="DAPI",
        z_slice=z_to_show,
        figsize=(6, 6),
        fig_kwargs={"dpi": plot_dpi},
    )

In [ ]:
# PolyT values to test
polyt_values = [60, 150, 285, 500]

In [ ]:
for polyt_size in polyt_values:
    layer_name = f"minmax_polyt_{polyt_size}"

    print(f"PolyT filter size = {polyt_size}")

    sdata = sp.im.min_max_filtering(
        sdata=sdata,
        img_layer=INPUT_LAYER,
        output_layer=layer_name,
        crd=coordinates,
        size_min_max_filter=[135, polyt_size],
        chunks=4096,
        overwrite=True,
    )

    sp.pl.plot_shapes(
        sdata,
        img_layer=[layer_name],
        crd=coordinates,
        channel="PolyT",
        z_slice=z_to_show,
        figsize=(6, 6),
        fig_kwargs={"dpi": plot_dpi},
    )

In [ ]:
MINMAX_LAYER = "min_max_filtered_DAPI_PolyT"

In [ ]:
sdata = sp.im.min_max_filtering(
    sdata=sdata,
    img_layer="Yujin_vizgen_Liver1Slice1_z_global",
    output_layer=MINMAX_LAYER,
    size_min_max_filter=[135, 285],   # DAPI, PolyT
    chunks=4096,
    overwrite=True,
)

In [ ]:
for i, z in enumerate(z_slices):
    print("=" * 60)
    print(f"Z layer {i} | z = {z}")

    for channel in ["DAPI", "PolyT"]:
        print(f"{channel}")

        sp.pl.plot_shapes(
            sdata,
            img_layer=[MINMAX_LAYER],
            crd=coordinates,
            channel=channel,
            z_slice=z,
            figsize=(6, 6),
            fig_kwargs={"dpi": plot_dpi},
        )

In [ ]:
dapi_clips = [2.0, 8.0, 18.5, 35.0, 60.0]

In [ ]:
for dapi_clip in dapi_clips:
    layer_name = f"clahe_dapi_{dapi_clip}"

    print(f"DAPI contrast_clip = {dapi_clip}")

    sdata = sp.im.enhance_contrast(
        sdata=sdata,
        img_layer=MINMAX_LAYER,
        output_layer=layer_name,
        contrast_clip=[dapi_clip, 13.5],
        crd=coordinates,
        chunks=1024,
        overwrite=True,
    )

    sp.pl.plot_shapes(
        sdata,
        img_layer=[layer_name],
        crd=coordinates,
        channel="DAPI",
        z_slice=z_to_show,
        figsize=(6, 6),
        fig_kwargs={"dpi": plot_dpi},
    )

In [ ]:
polyt_clips = [2.0, 6.0, 13.5, 25.0, 50.0]

In [ ]:
for polyt_clip in polyt_clips:
    layer_name = f"clahe_polyt_{polyt_clip}"

    print(f"PolyT contrast_clip = {polyt_clip}")

    sdata = sp.im.enhance_contrast(
        sdata=sdata,
        img_layer=MINMAX_LAYER,
        output_layer=layer_name,
        contrast_clip=[18.5, polyt_clip],
        crd=coordinates,
        chunks=1024,
        overwrite=True,
    )

    sp.pl.plot_shapes(
        sdata,
        img_layer=[layer_name],
        crd=coordinates,
        channel="PolyT",
        z_slice=z_to_show,
        figsize=(6, 6),
        fig_kwargs={"dpi": plot_dpi},
    )

In [ ]:
CLAHE_LAYER = "clahe_DAPI_PolyT"

In [ ]:
sdata = sp.im.enhance_contrast(
    sdata=sdata,
    img_layer=MINMAX_LAYER,
    output_layer=CLAHE_LAYER,
    contrast_clip=[18.5, 13.5],       # DAPI, PolyT
    chunks=4096,
    overwrite=True,
)

In [ ]:
for i, z in enumerate(z_slices):
    print("=" * 60)
    print(f"Z layer {i} | z = {z}")

    for channel in ["DAPI", "PolyT"]:

        print(f"{channel}")

        sp.pl.plot_shapes(
            sdata,
            img_layer=[CLAHE_LAYER],
            crd=coordinates,
            channel=channel,
            z_slice=z,
            figsize=(6, 6),
            fig_kwargs={"dpi": plot_dpi},
        )

In [ ]:
for i, z in enumerate(z_slices):
    print("=" * 60)
    print(f"Z layer {i} | z = {z}")

    for ch in ["DAPI", "PolyT"]:
        print(f"Channel: {ch}")

        sp.pl.plot_shapes(
            sdata,
            img_layer=["Yujin_vizgen_Liver1Slice1_z_global", "min_max_filtered_DAPI_PolyT", "clahe_DAPI_PolyT"],
            crd=coordinates,
            channel=ch,
            z_slice=z,
            figsize=(10, 10),
            fig_kwargs={"dpi": plot_dpi},
        )